# Why do some rules draw diagonal "crystal" lattices?

Experiment `2026-10-crystal-rules`. Written for a reader who has not seen the work. **Partial-programme status notebook**: S0, S3, S4, the local-patch test and the S1 ray anatomy are done; S1 (tail, ray ends) and S2 (transplant, ray-end knockouts, spinner isolation) are only partly done or not done (see README).

**Observation.** 30M-step renders of RLLLLRRL, LRRRRRLLR (and RLLLLLLLRRLRR) show thin straight 45-degree rays hundreds to ~1500 cells long. Candidate causes: a self-sustaining ray/highway on a ground (H-ray), a spinner launching the ant (H-spin), reversal/re-traversal (H-reverse), or ordinary frame-building (H-frame; README).

**Assumptions.** Blank board; deterministic rules, so no seed; a ray is a straight diagonal run (L = 128 steps, <= 8 steps/cell); only rays >= 64 cells counted (cutoff from controls); "motif" = smallest period at 98% self-agreement. Every prediction was registered in the README before its measurement; where I added an operationalisation or an exploratory step it is labelled.

In [1]:
import hashlib, pathlib, subprocess
res = pathlib.Path('results')
if not res.exists(): res = pathlib.Path('experiments/archive/2026-10-crystal-rules/results')
print('git HEAD (notebook run):', subprocess.run(['git','rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())
for f in ['s1_rays.txt','rays_calibration.txt','local_patch_summary.json','s1_bbox_octaves.csv','s4_crystal_scores.txt']:
    print(f'{f:28s}', hashlib.sha256((res/f).read_bytes()).hexdigest()[:16])
print('(raw ray CSVs are in gitignored dumps/; regenerate with examples/crystal_rules_rays.rs, command in README)')

git HEAD (notebook run): da88e61
s1_rays.txt                  78146acf1fb6fff2
rays_calibration.txt         9365c7ab23e4b20d
local_patch_summary.json     afaeb6de582de1d0
s1_bbox_octaves.csv          a01d216150d0b0a0
s4_crystal_scores.txt        7bdb6b67dd6041d9
(raw ray CSVs are in gitignored dumps/; regenerate with examples/crystal_rules_rays.rs, command in README)


## Instrument calibration for the ray segmenter (`rays_calibration.txt`)

In [2]:
print((res/'rays_calibration.txt').read_text())

# Ray-segmenter calibration (AGENTS.md rule 5), run before any crystal-rule ray data were inspected. git da88e61.
# Instrument: examples/crystal_rules_rays.rs (L=128 window, >=16 cells/window i.e. <=8 steps/cell, ||dx|-|dy||<=2).
# Commands (WSL, CARGO_TARGET_DIR=~/turmite-target): crystal_rules_rays {rw|rotor} SEED 4194304 8192 for SEED in 1 2 3; ant RLRRRL 4194304 8192; ant RL 131072 8192.
# Rays are counted by length in cells; "false" rays are anything found in controls that have no diagonal ballistic structure.
control             rays>=16cells  >=32  >=64  >=128  max_len_cells
random walk x3          10549        0     0      0       29
rotor x3                 2679        0     0      0       27
RLRRRL (frame builder)   4031        0     0      0       21
RL 2^17 steps               3        0     0      0       (all < 32 cells)
# RL at 2^22 steps leaves the 8192 board (highway): the first attempt panicked "left the board"; rerun at 2^17. RL's highway moves ~52 steps/cell,
# abov

**Shows.** False rays in a random walk, the rotor and the RLRRRL frame builder reach at most 29 cells, none reach 32; hence the 64-cell cutoff. Tests (`cargo test --release --example crystal_rules_rays`): planted-ray recovery, board loop equals reference ant, axial line is not a ray, motif rotation invariance; a deliberate threshold mutation made the planted test fail.
**Does not show.** That the segmenter finds every kind of crystal ray; the positive control is planted paths, not a known real crystal.

## S1: ray anatomy at 2^28 steps (`s1_rays.txt`)

In [3]:
print((res/'s1_rays.txt').read_text())

# s1 rays; commit da88e61b690507526ba4e2a2c017d9c7941beb91
# rays with length >= 64 cells; 2^28 steps, board 16384, blank start

## RLLLLRRL: rays >= 64 cells: 2960
  total ray length 741103 cells; median 179, max 1487; steps/cell median 2.09 (range 1.99-8.03)
  length log2-bins (count): 2^6:1099, 2^7:743, 2^8:787, 2^9:295, 2^10:36
  aperiodic rays: 90 (0.010 of length)
  motif p=  2 share 0.335 rays   945  04
  motif p=  2 share 0.261 rays   783  15
  motif p=  2 share 0.127 rays   366  26
  motif p=  2 share 0.027 rays    90  07
  motif p=  6 share 0.019 rays    45  166527
  motif p=  2 share 0.014 rays    46  46
  distinct motifs with >=1% of length: 8; dominant share 0.335, dominant p 2
  length-weighted reverse-overlap 0.091, same-heading overlap 0.352
  heading share of length: (+1,-1) 0.27, (-1,-1) 0.25, (-1,+1) 0.24, (+1,+1) 0.24
  terminating colour counts: 0:827, 5:417, 4:370, 6:358, 1:335, 2:230, 7:212, 3:211

## LRRRRRLLR: rays >= 64 cells: 254
  total ray length 43460 cell

**Shows (registered).** P2 (H-ray: one dominant motif, share >= 0.8, p in [4, 40]) is **falsified** for all three rules: 8, 20 and 23 colour motifs each carry >= 1% of ray length. P4 (H-reverse: reverse overlap <= 0.20) holds for RLLLLRRL (0.091) and barely for LRRRRRLLR (0.199); RLLLLLLLRRLRR is in between (0.375). No rule has > 0.5.
**Shows (exploratory, post hoc).** The colour motifs are mostly period-2 shifted copies; after mapping colours to turn letters the alternating staircase LR/LRLR carries 0.55-0.88 of ray length. So the registered motif definition (colour-level) splits one staircase object. The paired colours are opposite-turn colours (e.g. 04, 15, 26; 16, 68, 47), not colour 2; the S2 ground highway (50-86 steps/cell) is invisible to this segmenter (cap 8 steps/cell) and is a different object. The data suggest a staircase over a two-sublattice background of opposite-turn colours (not verified by printing colours around a ray); why it forms is open.
**Does not show.** What ends a ray, why the ground forms, or that rays are self-sustaining (that needs transplant/knockout interventions, S2).

## Earlier stages, from the README (numbers in `results/`)

In [4]:
print((res/'local_patch_summary.json').read_text()[:1500]); print(); print(*(res/'s1_bbox_octaves.csv').read_text().splitlines()[:12], sep=chr(10))

{
  "RLLLLRRL": {
    "pairs": 12,
    "colour_2_wins": 12,
    "prediction_passed": true,
    "input_sha256": "3b0b3033c7b7e4af2ae0bf99d515ee108ee9dd7e4149819273ec5485e7413146"
  },
  "LRRRRRLLR": {
    "pairs": 12,
    "colour_2_wins": 12,
    "prediction_passed": true,
    "input_sha256": "01f04d2d94897aacffc72524f405c31bd17000c974b471cfe1ad52a1b36f8fc6"
  },
  "RLLLLLLLRRLRR": {
    "pairs": 12,
    "colour_2_wins": 12,
    "prediction_passed": true,
    "input_sha256": "d361c36d376871fde813a92506413e78cb62ab2c769a2cd0dec50964212aa1bb"
  }
}


RLLLLRRL,4096,20,20
RLLLLRRL,8192,26,26
RLLLLRRL,16384,42,42
RLLLLRRL,32768,48,51
RLLLLRRL,65536,66,68
RLLLLRRL,131072,118,120
RLLLLRRL,262144,201,203
RLLLLRRL,524288,322,324
RLLLLRRL,1048576,503,505
RLLLLRRL,2097152,748,751
RLLLLRRL,4194304,951,954
RLLLLRRL,8388608,1330,1334


- **S0.** Renders reproduced; they are visit-count maps of a ~100% visited box with a uniform colour-2 ground (80-83% of visited cells). Prediction 1 as written was mis-specified; the replacement (diagonal-run cover of non-ground cells 0.96-0.98 vs 0.45 for RLRRRL) holds but its "score > 3" threshold failed (1.95-2.32).
- **S1 width.** Box width exponent 0.495 / 0.503 / 0.502 (2^20..2^28), prediction 3 held (H-frame); the control RLRRRL gives 0.499, so the exponent does not separate crystals from flat frame builders.
- **S2 (partial).** A lone ant on a uniform colour-2 ground is a true highway in all three rules (50 / 58 / 86 steps per diagonal cell), not on a blank board; local-patch test: with colour-2 exterior the ant travelled farther diagonally than with colour-0 in 12/12 paired windows per rule (prediction >= 9/12). Supports the ground as the medium; does not prove periodicity or reflection.
- **S3.** Prediction 6 falsified: all positions are sensitive, crystals are isolated under point mutation (0/142 Hamming-2); a family R L^k R R L is crystal-like for k = 4..15 (ground-fraction proxy).
- **S4 (census length <= 12).** Prediction 7 held at 87.5% (n = 8): max same-turn run >= 4 in 7 of 8 crystal rules; every crystal has ground colour 2 (2x visits); the necessary-not-sufficient half was not computed.

## Caveats
Three rules for S1 (one run each, deterministic); crystal definitions differ between stages (grid score vs ground-fraction proxy); ray endpoints good to ~128 steps; colour-to-letter collapse is my post hoc reading; S2 was run on a uniform board and a local patch, not a transplanted ray strip.

## Open (not run; stated by scope)
Ray-end knockouts, transplant onto reset/shuffled background, spinner isolation, exact periodicity of the ground highway, distance to the nearest earlier ray, why the sweep produces ground 2, and why a two-sublattice opposite-turn background forms behind the 2-steps/cell rays.